# MCP servers: sharing tools across apps

A small theatre wants its seat inventory reachable from three apps: the box-office chat assistant, a staff coding
tool and a lobby kiosk. In this notebook you write the theatre's seat server with the official MCP Python SDK,
connect a client to it, discover its tools, call them, break a call the way the kiosk did, plug the server into
the chat assistant as a host, and fix the error that hid the reason from the model.

About 10 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime, with no account and no key.
The model answers in this notebook are the lesson's recorded answers, replayed through the same code.

## 1. Install the MCP Python SDK and set up a short server log

The next cell adds the SDK version the lesson ran (`mcp` 2.3.0). The server and the client both run inside this
notebook (the SDK's in-memory mode), so you need no second program. The server writes a log, like the error log
(stderr) of a real server. The cell prints each log line as `SERVER LOG:`, with the exception names, so you can
see what the server knows and the client does not.

**Look for:** the line `ready`. pip may print a few lines about other packages; you can ignore them.

In [1]:
%pip install -q mcp==2.3.0

import json, logging
from mcp import Client, MCPError
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError


class ServerLog(logging.Handler):
    def emit(self, record):
        print("SERVER LOG:", record.getMessage())
        exc = record.exc_info[1] if record.exc_info else None
        chain = []
        while exc is not None:  # the exception and the one that caused it
            chain.append(f"{type(exc).__name__}: {exc}")
            exc = exc.__cause__
        for line in reversed(chain):
            print("   ", line)


server_log = logging.getLogger("mcp")
server_log.handlers = [ServerLog()]
server_log.setLevel(logging.INFO)
server_log.propagate = False
print("ready")

Note: you may need to restart the kernel to use updated packages.


ready


## 2. The seat server

This is the lesson's server, line for line: the seat data, one resource (the performances on sale) and two tools.
There is no JSON-RPC here and no schema written by hand. The function name becomes the tool's name, the
docstring its description, and the type hints its input schema.

**Look for:** the printed seat table. F9 is `held` and F6 and G8 are `sold`.

In [2]:
SEATS = {  # performance id -> row -> seat -> "free" | "held" | "sold"
    "HAM-1128": {"F": {"F6": "sold", "F7": "free", "F8": "free", "F9": "held"},
                 "G": {"G6": "free", "G7": "free", "G8": "sold", "G9": "free"}},
}
PERFORMANCES = [{"performance_id": "HAM-1128", "title": "Hamlet", "date": "2026-11-28", "starts": "19:30"}]

mcp = MCPServer("theatre-seats")


@mcp.resource("seats://performances")
def performances() -> str:
    """The performances on sale, with their ids."""
    return json.dumps(PERFORMANCES)


@mcp.tool()
def find_free_seats(performance_id: str, row: str) -> list[str]:
    """List the free seats in one row of one performance."""
    return [s for s, state in SEATS[performance_id][row].items() if state == "free"]


@mcp.tool()
def hold_seats(performance_id: str, seats: list[str]) -> dict:
    """Hold seats for 10 minutes while the customer pays."""
    rows = SEATS[performance_id]
    taken = [s for s in seats if rows[s[0]][s] != "free"]
    if taken:
        raise ValueError(f"Not free: {', '.join(taken)}")
    for s in seats:
        rows[s[0]][s] = "held"
    return {"held": seats, "minutes": 10}


for row, seats in SEATS["HAM-1128"].items():
    print("row", row, seats)

row F {'F6': 'sold', 'F7': 'free', 'F8': 'free', 'F9': 'held'}
row G {'G6': 'free', 'G7': 'free', 'G8': 'sold', 'G9': 'free'}


## 3. Discovery: ask the server what it offers

A client connects and asks: which tools (`tools/list`), which resources (`resources/list`). Then it reads the one
resource (`resources/read`). The client did not know any tool name before it asked.

**Look for:** two tool names, `find_free_seats` and `hold_seats`. In the input schema of find_free_seats,
`required` is `["performance_id", "row"]`: it was built from the type hints. The resource text has the id
`HAM-1128`.

In [3]:
async with Client(mcp) as client:
    tools = (await client.list_tools()).tools
    resources = (await client.list_resources()).resources
    shows = (await client.read_resource("seats://performances")).contents[0].text

print("TOOLS:", [t.name for t in tools])
find = tools[0]
print("DESCRIPTION OF", find.name + ":", find.description)
print("INPUT SCHEMA:", json.dumps(find.input_schema))
print("REQUIRED:", find.input_schema["required"])
print("RESOURCES:", [str(r.uri) for r in resources])
print("PERFORMANCES:", shows)

TOOLS: ['find_free_seats', 'hold_seats']
DESCRIPTION OF find_free_seats: List the free seats in one row of one performance.
INPUT SCHEMA: {"type": "object", "properties": {"performance_id": {"title": "Performance Id", "type": "string"}, "row": {"title": "Row", "type": "string"}}, "required": ["performance_id", "row"], "title": "find_free_seatsArguments"}
REQUIRED: ['performance_id', 'row']
RESOURCES: ['seats://performances']
PERFORMANCES: [{"performance_id": "HAM-1128", "title": "Hamlet", "date": "2026-11-28", "starts": "19:30"}]


## 4. A call: free seats in row G

The kiosk's client sends `tools/call` with the tool name and its arguments. The answer is a result with
`isError` and the content.

**Look for:** `is_error = False` and the seats `['G6', 'G7', 'G9']` (G8 is sold).

In [4]:
async with Client(mcp) as client:
    ok = await client.call_tool("find_free_seats", {"performance_id": "HAM-1128", "row": "G"})

print("ROW G: is_error =", ok.is_error, "| structured_content =", ok.structured_content)
print("TEXT CONTENT:", [c.text for c in ok.content])

ROW G: is_error = False | structured_content = {'result': ['G6', 'G7', 'G9']}
TEXT CONTENT: ['G6', 'G7', 'G9']


## 5. Broken on purpose: the kiosk's old button

The kiosk's "show free seats" button was written against an older version of the server, whose parameter was
called `performance`. The server now calls it `performance_id`, and step 3 showed that schema. The button sends
the old name anyway, typed into its code.

**Look for:** `is_error = True`. The answer is still a normal result (no exception in this cell), and its text
names `performance_id` and says `Field required`. The server log says it rejected the arguments.

In [5]:
async with Client(mcp) as client:
    # the button's hard-coded call, from the old version of the server
    bad = await client.call_tool("find_free_seats", {"performance": "HAM-1128", "row": "F"})

print("BUTTON: is_error =", bad.is_error)
print("TEXT:", bad.content[0].text)

SERVER LOG: Tool 'find_free_seats' rejected arguments: ['performance_id']
BUTTON: is_error = True
TEXT: Error executing tool find_free_seats: 1 validation error for find_free_seatsArguments
performance_id
  Field required [type=missing, input_value={'performance': 'HAM-1128', 'row': 'F'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


## 6. Two kinds of failure

The lesson's prediction screen, run for real on the same server: free seats in row F; hold F8 and F9 (F9 is
held, so `hold_seats` raises an exception before it holds anything); row F again; and a read of a resource the
server does not have. The last one raises an `MCPError` in the client, so it is wrapped in `try/except`.

**Look for:**
- Row F gives `['F7', 'F8']`, before and after the failed hold: nothing was held.
- The hold gives `is_error = True` and only the text `Error executing tool hold_seats`. No seat, no reason.
- The server log has the reason, `ValueError: Not free: F9`. It stayed on the server.
- The unknown resource gives a JSON-RPC error, code `-32602`, `Unknown resource: seats://matinees`.

In [6]:
async with Client(mcp) as client:
    row_f = await client.call_tool("find_free_seats", {"performance_id": "HAM-1128", "row": "F"})
    print("1 ROW F:", row_f.is_error, row_f.structured_content)
    hold = await client.call_tool("hold_seats", {"performance_id": "HAM-1128", "seats": ["F8", "F9"]})
    print("2 HOLD F8 F9:", hold.is_error, [c.text for c in hold.content])
    row_f2 = await client.call_tool("find_free_seats", {"performance_id": "HAM-1128", "row": "F"})
    print("3 ROW F AGAIN:", row_f2.is_error, row_f2.structured_content)
    try:
        gone = await client.read_resource("seats://matinees")
        print("4 READ seats://matinees:", gone)
    except MCPError as e:
        print("4 READ seats://matinees: MCPError", e.error.code, e.error.message)

1 ROW F: False {'result': ['F7', 'F8']}
SERVER LOG: Tool 'hold_seats' raised an unexpected exception
    ValueError: Not free: F9
    UnexpectedToolError: Error executing tool hold_seats
2 HOLD F8 F9: True ['Error executing tool hold_seats']
3 ROW F AGAIN: False {'result': ['F7', 'F8']}
SERVER LOG: Resource 'seats://matinees' failed: 'Unknown resource: seats://matinees'
4 READ seats://matinees: MCPError -32602 Unknown resource: seats://matinees


## 7. The host: turn tools/list into the model's tools

The box-office chat assistant is now an MCP host. At start-up it lists the server's tools and converts each one
into a tool for the model with `as_model_tool`. It reads the performances resource and puts its text at the end
of the system prompt. This is the request `host.py` sent to the model in the lesson.

**Look for:** the two tool names sent to the model, and a system prompt that ends with the resource text, where
`HAM-1128` comes from. The host has no code for any single tool.

In [7]:
MODEL = "google/gemini-2.5-flash-lite"
SYSTEM = ("You are the box-office assistant of a small theatre. Use the tools to check and hold seats. "
          "Never promise a seat the tools did not confirm. Performances on sale: ")
USER = "Hi, I'd like seats F8 and F9 for Hamlet on 28 November. Can you hold them for me?"


def as_model_tool(t):  # one MCP tool (from tools/list) -> one OpenAI-style tool for the model
    return {"type": "function", "function": {"name": t.name, "description": t.description, "parameters": t.input_schema}}


async def first_request(server):
    async with Client(server) as client:
        tools = (await client.list_tools()).tools
        shows = (await client.read_resource("seats://performances")).contents[0].text
    return {"model": MODEL, "temperature": 0, "max_tokens": 200,
            "messages": [{"role": "system", "content": SYSTEM + shows}, {"role": "user", "content": USER}],
            "tools": [as_model_tool(t) for t in tools]}


request = await first_request(mcp)
print("TOOLS SENT TO THE MODEL:", [t["function"]["name"] for t in request["tools"]])
print("SYSTEM:", request["messages"][0]["content"])

TOOLS SENT TO THE MODEL: ['find_free_seats', 'hold_seats']
SYSTEM: You are the box-office assistant of a small theatre. Use the tools to check and hold seats. Never promise a seat the tools did not confirm. Performances on sale: [{"performance_id": "HAM-1128", "title": "Hamlet", "date": "2026-11-28", "starts": "19:30"}]


## 8. The model's answers, recorded

You have no API key here, so the model's answers are the lesson's recordings: a small hosted model
(google/gemini-2.5-flash-lite, temperature 0), recorded on 2026-11-13, copied exactly and trimmed to the
message. `ask_model` takes the same request a live call would take and replays the recording made for that
conversation. It matches on the last message.

**Look for:** three recordings. The first one is a tool request for `hold_seats`, with `content` None.

In [8]:
RECORDED = [  # (role, content of the last message sent) -> the model's reply message
    {"name": "hold-1", "last": ("user", USER),
     "reply": {"role": "assistant", "content": None,
               "tool_calls": [{"type": "function", "index": 0, "id": "tool_hold_seats_O0xpRpixGKktMP22p8x7",
                               "function": {"name": "hold_seats",
                                            "arguments": "{\"seats\":[\"F8\",\"F9\"],\"performance_id\":\"HAM-1128\"}"}}]}},
    {"name": "hold-2-crash", "last": ("tool", "Error executing tool hold_seats"),
     "reply": {"role": "assistant",
               "content": "I am sorry, I cannot fulfill this request. The available tool `hold_seats` is not working correctly."}},
    {"name": "hold-2-fixed", "last": ("tool", "Error executing tool hold_seats: Not free: F9"),
     "reply": {"role": "assistant",
               "content": "I am sorry, but it seems that seat F9 is no longer available. We can still hold seat F8 for you, would you like to proceed?"}},
]


def ask_model(request):
    last = request["messages"][-1]
    for rec in RECORDED:
        if (last["role"], last["content"]) == rec["last"]:
            print(f"[replaying recording {rec['name']}]")
            return rec["reply"]
    print("[no recording for this conversation: the model was never asked this]")
    return None


for rec in RECORDED:
    print(rec["name"], "->", json.dumps(rec["reply"])[:110], "...")

hold-1 -> {"role": "assistant", "content": null, "tool_calls": [{"type": "function", "index": 0, "id": "tool_hold_seats_ ...
hold-2-crash -> {"role": "assistant", "content": "I am sorry, I cannot fulfill this request. The available tool `hold_seats` i ...
hold-2-fixed -> {"role": "assistant", "content": "I am sorry, but it seems that seat F9 is no longer available. We can still h ...


## 9. One turn of the host's loop, with the crash

The host sends the request, gets the model's tool request, forwards it to the server as `tools/call`, and sends
the server's text back to the model as a tool message. It passes the server's text on unchanged.

**Look for:** the model asked for `hold_seats` with F8 and F9. The server said `is_error = True` with only
`Error executing tool hold_seats`, while the server log has `Not free: F9`. The model then tells the customer
the tool is not working, because it never got the reason.

In [9]:
async def one_turn(server):
    request = await first_request(server)
    reply = ask_model(request)
    call = reply["tool_calls"][0]
    args = json.loads(call["function"]["arguments"])
    print("MODEL ASKED:", call["function"]["name"], call["function"]["arguments"])
    async with Client(server) as client:
        result = await client.call_tool(call["function"]["name"], args)
    text = "\n".join(c.text for c in result.content)
    print("SERVER SAID: is_error =", result.is_error, "| text =", json.dumps(text))
    request["messages"] += [{"role": "assistant", "content": reply.get("content"), "tool_calls": reply["tool_calls"]},
                            {"role": "tool", "tool_call_id": call["id"], "content": text}]
    answer = ask_model(request)
    print("MODEL TOLD THE CUSTOMER:", answer["content"] if answer else None)


await one_turn(mcp)

[replaying recording hold-1]
MODEL ASKED: hold_seats {"seats":["F8","F9"],"performance_id":"HAM-1128"}
SERVER LOG: Tool 'hold_seats' raised an unexpected exception
    ValueError: Not free: F9
    UnexpectedToolError: Error executing tool hold_seats
SERVER SAID: is_error = True | text = "Error executing tool hold_seats"
[replaying recording hold-2-crash]
MODEL TOLD THE CUSTOMER: I am sorry, I cannot fulfill this request. The available tool `hold_seats` is not working correctly.


## 10. The fix: raise ToolError

An ordinary exception is "unexpected" to the SDK: the client gets only `Error executing tool hold_seats`, and
the message stays in the server's log. An error you expect has its own type, `ToolError`, and its message is
sent. The next cell builds the fixed server: the same resource and `find_free_seats`, and `hold_seats` with one
line changed. Then it runs the same turn of the host's loop.

**Look for:** the same request now comes back with `Error executing tool hold_seats: Not free: F9`, still with
`is_error = True`. The model now tells the customer that F9 is no longer available and offers F8.

In [10]:
mcp_fixed = MCPServer("theatre-seats")
mcp_fixed.resource("seats://performances")(performances)
mcp_fixed.tool()(find_free_seats)


@mcp_fixed.tool()
def hold_seats(performance_id: str, seats: list[str]) -> dict:
    """Hold seats for 10 minutes while the customer pays."""
    rows = SEATS[performance_id]
    taken = [s for s in seats if rows[s[0]][s] != "free"]
    if taken:
        raise ToolError(f"Not free: {', '.join(taken)}")  # was: raise ValueError(...)
    for s in seats:
        rows[s[0]][s] = "held"
    return {"held": seats, "minutes": 10}


await one_turn(mcp_fixed)

[replaying recording hold-1]
MODEL ASKED: hold_seats {"seats":["F8","F9"],"performance_id":"HAM-1128"}
SERVER LOG: Tool 'hold_seats' failed: 'Error executing tool hold_seats: Not free: F9'
SERVER SAID: is_error = True | text = "Error executing tool hold_seats: Not free: F9"
[replaying recording hold-2-fixed]
MODEL TOLD THE CUSTOMER: I am sorry, but it seems that seat F9 is no longer available. We can still hold seat F8 for you, would you like to proceed?


## 11. Optional: your own API key

Skip this cell; nothing below needs it. If you have an OpenRouter key, set it in Colab's Secrets panel as
`OPENROUTER_API_KEY` (or as an environment variable), and this cell sends the step 7 request to the same model
live. Its answer may differ a little from the recording. With no key, it prints one line and does nothing.

In [11]:
import os, urllib.request

key = os.environ.get("OPENROUTER_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        key = None
if not key:
    print("No API key set: skipped. The recordings above are what the lesson used.")
else:
    req = urllib.request.Request("https://openrouter.ai/api/v1/chat/completions", data=json.dumps(request).encode(),
                                 headers={"Authorization": "Bearer " + key, "Content-Type": "application/json"})
    live = json.load(urllib.request.urlopen(req, timeout=60))
    print("LIVE REPLY:", json.dumps(live["choices"][0]["message"], indent=1))

No API key set: skipped. The recordings above are what the lesson used.


## Now change it

Three small experiments. In each cell, change the line marked `# <- CHANGE THIS`, run the cell, and compare.

### Experiment 1: the docstring is what the model reads

Change the docstring of `find_free_seats` to your own sentence, for example
`"Free seats in one row. Use before hold_seats."`. The host's `as_model_tool` is not touched.

**What should change:** the description in the tool list the model gets is now your sentence. The server sends
whatever the docstring says, and the host passes it on, so the docstring is written for the model.

In [12]:
mcp_exp1 = MCPServer("theatre-seats")


@mcp_exp1.tool()
def find_free_seats(performance_id: str, row: str) -> list[str]:
    """List the free seats in one row of one performance."""  # <- CHANGE THIS
    return [s for s, state in SEATS[performance_id][row].items() if state == "free"]


async with Client(mcp_exp1) as client:
    tools = (await client.list_tools()).tools
print("DESCRIPTION THE MODEL GETS:", as_model_tool(tools[0])["function"]["description"])

DESCRIPTION THE MODEL GETS: List the free seats in one row of one performance.


### Experiment 2: a hold that works

Change the seats to two free ones, `["F7", "F8"]`. Nothing else changes.

**What should change:** the hold now returns `is_error = False` with `{'held': ['F7', 'F8'], 'minutes': 10}`,
and row F afterwards has no free seats left, `[]`: a successful call changes the server's state. The first line
of the cell puts row F back as it was, so you can run it again.

In [13]:
SEATS["HAM-1128"]["F"] = {"F6": "sold", "F7": "free", "F8": "free", "F9": "held"}  # row F as at the start
want = ["F8", "F9"]  # <- CHANGE THIS

async with Client(mcp_fixed) as client:
    hold = await client.call_tool("hold_seats", {"performance_id": "HAM-1128", "seats": want})
    print("HOLD", want, ": is_error =", hold.is_error, "|", hold.content[0].text if hold.is_error else json.loads(hold.content[0].text))
    row_f = await client.call_tool("find_free_seats", {"performance_id": "HAM-1128", "row": "F"})
    print("ROW F AFTER:", row_f.structured_content)

SERVER LOG: Tool 'hold_seats' failed: 'Error executing tool hold_seats: Not free: F9'
HOLD ['F8', 'F9'] : is_error = True | Error executing tool hold_seats: Not free: F9
ROW F AFTER: {'result': ['F7', 'F8']}


### Experiment 3: rename a parameter, as the server team did

Rename the parameter `row` to `row_letter` in the `def` line, and in the line under the docstring, too. The
client below keeps sending `row`, typed into its code, like the kiosk.

**What should change:** the schema's `required` list now says `row_letter`, and the hard-coded call fails with
`is_error = True` and `Field required` for `row_letter`. A host that builds calls from the schema, like the chat
assistant, would pass the new name to its model and keep working.

In [14]:
mcp_exp3 = MCPServer("theatre-seats")


@mcp_exp3.tool()
def find_free_seats(performance_id: str, row: str) -> list[str]:  # <- CHANGE THIS
    """List the free seats in one row of one performance."""
    return [s for s, state in SEATS[performance_id][row].items() if state == "free"]


async with Client(mcp_exp3) as client:
    tools = (await client.list_tools()).tools
    print("REQUIRED:", tools[0].input_schema["required"])
    res = await client.call_tool("find_free_seats", {"performance_id": "HAM-1128", "row": "G"})  # hard-coded, like the kiosk
print("HARD-CODED CALL: is_error =", res.is_error, "|", res.structured_content or res.content[0].text.splitlines()[:3])

REQUIRED: ['performance_id', 'row']
HARD-CODED CALL: is_error = False | {'result': ['G6', 'G7', 'G9']}


## What you should see

- Step 3: `list_tools()` prints two names, `find_free_seats` and `hold_seats`, and the input schema of
  find_free_seats has `required: ["performance_id", "row"]`, built from the type hints.
- Step 4: find_free_seats for HAM-1128, row G returns `['G6', 'G7', 'G9']` with `is_error = False`.
- Step 5: calling it with `performance` instead of `performance_id` returns `is_error = True`, and the text names
  `performance_id`, "Field required".
- Steps 6 and 9: hold_seats for F8 and F9 returns only "Error executing tool hold_seats", while the server log
  has `ValueError: Not free: F9`.
- Step 10: after switching to `ToolError` it returns "Error executing tool hold_seats: Not free: F9", and the
  replayed model answer changes from "the tool is not working" to "F9 is no longer available".